# Limpeza de `amizades.csv`

Uma relação de seguir por linha (Ana segue Bruno e Bruno segue Ana são duas linhas). Saem três tabelas em `dados/processados/`:

| Tabela | Uma linha por | Para quê |
|---|---|---|
| `amizades_limpa.parquet` | relação de seguir até o corte | base para qualquer análise de rede |
| `rede_por_conta.parquet` | conta cadastrada até o corte (sem contas de teste) | tamanho da rede de cada pessoa no corte, inclusive quantos dos seus seguidores pagam |
| `rede_mes.parquet` | assinatura × mês, as mesmas linhas do painel `assinaturas_mes` | rede acumulada até o dia 1 de cada mês, para o modelo usar só o que existia na data |

**Regras** (no `decisoes.md`):
1. Relações com `desde` depois de 30/09/2026 saem (informação posterior ao corte).
2. O arquivo parece um retrato: se deixar de seguir apaga a linha, a rede de meses passados aparece maior do que era.
   Não há como corrigir com este arquivo; a ressalva fica registrada.

Diagnóstico feito antes da limpeza (não havia notebook de qualidade para este arquivo): as checagens estão nas seções 0 e 1.
Nada em `dados/brutos/` é alterado: o hash é conferido no início e no fim.

In [1]:
import hashlib
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 180)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)

ARQUIVOS = ["amizades.csv", "usuarios.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

registro = []

def registrar(tabela, passo, n, obs=""):
    registro.append({"tabela": tabela, "passo": passo, "linhas": n, "observacao": obs})
    print(f"[{tabela}] {passo}: {n:,} linhas" + (f" ({obs})" if obs else ""))

## 0. Leitura e checagens de qualidade

Todas deram zero no reconhecimento; ficam aqui como `assert` para a regra valer se o arquivo mudar.

In [2]:
am = pd.read_csv(BRUTOS / "amizades.csv", dtype=str, keep_default_na=False)
usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, keep_default_na=False)
assin = pd.read_parquet(PROCESSADOS / "assinaturas_limpa.parquet")
painel = pd.read_parquet(PROCESSADOS / "assinaturas_mes.parquet", columns=["assinatura_id", "user_id", "mes"])

assert len(am) == 113_480   # cabeçalho da seção 04 do dicionário

u = usuarios.set_index("user_id")
email = u["email"].str.strip()
cadastro = pd.to_datetime(u["data_cadastro"], format="%Y-%m-%d %H:%M:%S")
teste = email.str.lower().str.endswith("@vitaliza.com.br")
duplicada = email.str.contains(r"[A-Z]") & email.str.lower().duplicated(keep=False)
am["desde_data"] = pd.to_datetime(am["desde"], format="%Y-%m-%d")

checagens = {}
for col in ["seguidor_id", "seguido_id"]:
    checagens[f"{col} vazio"] = (am[col].str.strip() == "").sum()
    checagens[f"{col} fora do padrão u_xxxxxxxx"] = (~am[col].str.fullmatch(r"u_[0-9a-f]{8}")).sum()
    checagens[f"{col} sem cadastro"] = (~am[col].isin(u.index)).sum()
    checagens[f"{col} conta de teste"] = am[col].map(teste).sum()
    checagens[f"{col} conta duplicada (e-mail com maiúscula)"] = am[col].map(duplicada).sum()
    checagens[f"desde antes do cadastro do {col.split('_')[0]}"] = (am["desde_data"] < am[col].map(cadastro).dt.normalize()).sum()
checagens["conta que segue a si mesma"] = (am["seguidor_id"] == am["seguido_id"]).sum()
checagens["par seguidor-seguido repetido"] = am.duplicated(["seguidor_id", "seguido_id"]).sum()
checagens["desde que não converte em data"] = am["desde_data"].isna().sum()

tabela = pd.Series(checagens, name="linhas").to_frame()
display(tabela)
assert tabela["linhas"].sum() == 0
registrar("amizades", "0. Arquivo bruto", len(am), "0 nas checagens de completude, validade e unicidade")

,linhas
seguidor_id vazio,0
seguidor_id fora do padrão u_xxxxxxxx,0
seguidor_id sem cadastro,0
seguidor_id conta de teste,0
seguidor_id conta duplicada (e-mail com maiúscula),0
desde antes do cadastro do seguidor,0
seguido_id vazio,0
seguido_id fora do padrão u_xxxxxxxx,0
seguido_id sem cadastro,0
seguido_id conta de teste,0


[amizades] 0. Arquivo bruto: 113,480 linhas (0 nas checagens de completude, validade e unicidade)


## 1. Temporalidade

O volume por ano acompanha os cadastros (cerca de 1,9 relação por conta nova em todos os anos): o salto de 2024 é
crescimento da base. O arquivo vai até 09/10/2026, depois do corte.

In [3]:
por_ano = pd.DataFrame({
    "relações": am["desde_data"].dt.year.value_counts().sort_index(),
    "cadastros": cadastro.dt.year.value_counts().sort_index(),
})
por_ano["relações por cadastro"] = (por_ano["relações"] / por_ano["cadastros"]).round(2)
display(por_ano.T)

pos_corte = am["desde_data"] > CORTE
print(f"desde: {am['desde_data'].min().date()} a {am['desde_data'].max().date()}; depois do corte: {pos_corte.sum()}")
display(am.loc[pos_corte, "desde_data"].dt.date.value_counts().sort_index().rename("relações").to_frame().T)
cad_pos = pd.concat([am.loc[pos_corte, c].map(cadastro) for c in ["seguidor_id", "seguido_id"]])
print(f"dessas, com conta cadastrada depois do corte: {(cad_pos > CORTE + pd.Timedelta(days=1)).sum()}")
assert pos_corte.sum() == 273

,2020,2021,2022,2023,2024,2025,2026,2027
relações,337.00,2596.00,2430.00,2745.0,16446.00,48415.00,40511.00,NaN
cadastros,439.00,1383.00,1336.00,1441.0,11939.00,26167.00,22202.00,41.0
relações por cadastro,0.77,1.88,1.82,1.9,1.38,1.85,1.82,NaN


desde: 2020-09-05 a 2026-10-09; depois do corte: 273


desde_data,2026-10-01,2026-10-02,2026-10-03,2026-10-04,2026-10-05,2026-10-06,2026-10-07,2026-10-08,2026-10-09
relações,54,48,48,35,39,15,21,10,3


dessas, com conta cadastrada depois do corte: 0


## 2. Limpeza

In [4]:
limpa = am.loc[~pos_corte, ["seguidor_id", "seguido_id", "desde_data"]].rename(columns={"desde_data": "desde"})
limpa = limpa.sort_values(["desde", "seguidor_id", "seguido_id"]).reset_index(drop=True)
registrar("amizades", "1. Até o corte", len(limpa), f"{pos_corte.sum()} removidas (desde > 30/09/2026)")

volta = limpa.merge(limpa, left_on=["seguidor_id", "seguido_id"], right_on=["seguido_id", "seguidor_id"],
                    suffixes=("", "_volta"))
limpa["reciproca"] = limpa.set_index(["seguidor_id", "seguido_id"]).index.isin(
    volta.set_index(["seguidor_id", "seguido_id"]).index)
dias_volta = (volta["desde"] - volta["desde_volta"]).dt.days.abs()
print(f"relações recíprocas: {limpa['reciproca'].sum():,} ({limpa['reciproca'].mean():.0%}); "
      f"a segunda vem até {dias_volta.max()} dias depois da primeira")
assert limpa["reciproca"].sum() % 2 == 0
registrar("amizades", "2. Marca de reciprocidade", len(limpa), f"{limpa['reciproca'].sum():,} recíprocas")

[amizades] 1. Até o corte: 113,207 linhas (273 removidas (desde > 30/09/2026))


relações recíprocas: 84,498 (75%); a segunda vem até 9 dias depois da primeira
[amizades] 2. Marca de reciprocidade: 113,207 linhas (84,498 recíprocas)


## 3. Rede de cada conta numa data

`rede_em(d)` conta, para cada conta, as relações formadas **antes** do instante `d`:
seguidores, contas que segue, relações mútuas (as duas direções já existiam) e quantos seguidores e seguidos
eram assinantes ativos no corte. A mútua conta a partir da segunda direção.

In [5]:
mutuas = volta.loc[volta["seguidor_id"] < volta["seguido_id"]].assign(
    formada=lambda x: x[["desde", "desde_volta"]].max(axis=1))
mutuas_por_conta = pd.concat([
    mutuas[["seguidor_id", "formada"]].rename(columns={"seguidor_id": "user_id"}),
    mutuas[["seguido_id", "formada"]].rename(columns={"seguido_id": "user_id"}),
])
assert len(mutuas_por_conta) == limpa["reciproca"].sum()

ativos_corte = set(assin.loc[assin["ativa_no_corte"], "user_id"])

def rede_em(d):
    r = limpa.loc[limpa["desde"] < d]
    m = mutuas_por_conta.loc[mutuas_por_conta["formada"] < d]
    return pd.DataFrame({
        "n_seguidores": r.groupby("seguido_id").size(),
        "n_seguindo": r.groupby("seguidor_id").size(),
        "n_mutuas": m.groupby("user_id").size(),
    }).fillna(0).astype("int64")

## 4. Tabela: rede por conta no corte (`rede_por_conta.parquet`)

Uma linha por conta cadastrada até o corte, sem as contas de teste. Conta sem relação fica com zero.

In [6]:
contas = u.loc[~teste & (cadastro < FIM_CORTE)].index
print(f"contas: {len(u):,} no cadastro; {teste.sum()} de teste e {(cadastro >= FIM_CORTE).sum()} cadastradas depois do corte "
      f"ficam fora -> {len(contas):,}")

rpc = rede_em(FIM_CORTE).reindex(contas, fill_value=0)
rpc.index.name = "user_id"
seg = limpa.assign(ativo=limpa["seguidor_id"].isin(ativos_corte)).groupby("seguido_id")["ativo"].sum()
sgd = limpa.assign(ativo=limpa["seguido_id"].isin(ativos_corte)).groupby("seguidor_id")["ativo"].sum()
rpc["n_seguidores_assinantes_ativos"] = seg.reindex(contas, fill_value=0).astype("int64")
rpc["n_seguindo_assinantes_ativos"] = sgd.reindex(contas, fill_value=0).astype("int64")
primeira = pd.concat([limpa[["seguidor_id", "desde"]].rename(columns={"seguidor_id": "user_id"}),
                      limpa[["seguido_id", "desde"]].rename(columns={"seguido_id": "user_id"})]).groupby("user_id")["desde"].min()
rpc["primeira_conexao"] = primeira.reindex(contas)
rpc["assinante_ativo_no_corte"] = rpc.index.isin(ativos_corte)
rpc = rpc.reset_index()

assert rpc["n_seguidores"].sum() == rpc["n_seguindo"].sum() == len(limpa)
assert rpc["n_mutuas"].sum() == limpa["reciproca"].sum()
sem_rede = (rpc[["n_seguidores", "n_seguindo"]].sum(axis=1) == 0)
registrar("rede_por_conta", "3. Uma linha por conta", len(rpc), f"{sem_rede.sum():,} sem nenhuma relação")

display(rpc.groupby("assinante_ativo_no_corte")[["n_seguidores", "n_seguindo", "n_mutuas",
                                                   "n_seguidores_assinantes_ativos"]].mean().round(2))

contas: 64,948 no cadastro; 60 de teste e 119 cadastradas depois do corte ficam fora -> 64,769


[rede_por_conta] 3. Uma linha por conta: 64,769 linhas (10,607 sem nenhuma relação)


,n_seguidores,n_seguindo,n_mutuas,n_seguidores_assinantes_ativos
assinante_ativo_no_corte,,,,
False,1.77,1.77,1.32,0.37
True,1.66,1.67,1.24,0.43


## 5. Tabela: rede por assinatura e mês (`rede_mes.parquet`)

As mesmas linhas do painel: a rede da conta formada antes do dia 1 do mês.

In [7]:
partes = []
for mes, grupo in painel.groupby("mes"):
    r = rede_em(mes).reindex(grupo["user_id"].values, fill_value=0)
    partes.append(pd.concat([grupo.reset_index(drop=True), r.reset_index(drop=True)], axis=1))
rm = pd.concat(partes, ignore_index=True)
rm = painel.merge(rm, on=["assinatura_id", "user_id", "mes"], how="left", validate="one_to_one")
assert len(rm) == len(painel) and rm[["n_seguidores", "n_seguindo", "n_mutuas"]].notna().all().all()

rm = rm.sort_values(["assinatura_id", "mes"]).reset_index(drop=True)
diminui = (rm.groupby("assinatura_id")[["n_seguidores", "n_seguindo", "n_mutuas"]].diff() < 0).any(axis=1).sum()
print(f"linhas em que a rede diminui de um mês para o outro: {diminui}")
assert diminui == 0

# conferência: rede no corte = a do resumo por conta, para as assinaturas ativas
ativas = assin.loc[assin["ativa_no_corte"], "user_id"]
no_corte = rede_em(FIM_CORTE).reindex(ativas.values, fill_value=0)
assert (no_corte.values == rpc.set_index("user_id").loc[ativas.values, ["n_seguidores", "n_seguindo", "n_mutuas"]].values).all()
registrar("rede_mes", "4. Uma linha por linha do painel", len(rm),
          f"{(rm['n_seguidores'] + rm['n_seguindo'] > 0).sum():,} com alguma relação")
display(rm.groupby("mes")[["n_seguidores", "n_seguindo", "n_mutuas"]].mean().round(2).T)

linhas em que a rede diminui de um mês para o outro: 0


[rede_mes] 4. Uma linha por linha do painel: 327,771 linhas (240,568 com alguma relação)


mes,2024-07-01,2024-08-01,2024-09-01,2024-10-01,2024-11-01,2024-12-01,2025-01-01,2025-02-01,2025-03-01,2025-04-01,2025-05-01,2025-06-01,2025-07-01,2025-08-01,2025-09-01,2025-10-01,2025-11-01,2025-12-01,2026-01-01,2026-02-01,2026-03-01,2026-04-01,2026-05-01,2026-06-01,2026-07-01,2026-08-01,2026-09-01
n_seguidores,1.63,1.47,1.41,1.39,1.44,1.50,1.46,1.45,1.44,1.49,1.53,1.57,1.58,1.58,1.55,1.52,1.53,1.56,1.51,1.49,1.49,1.52,1.55,1.58,1.60,1.58,1.56
n_seguindo,1.63,1.47,1.39,1.36,1.40,1.47,1.43,1.43,1.43,1.47,1.51,1.55,1.57,1.57,1.55,1.52,1.53,1.56,1.51,1.49,1.49,1.52,1.55,1.59,1.61,1.59,1.57
n_mutuas,1.21,1.09,1.03,1.01,1.05,1.10,1.07,1.07,1.06,1.10,1.13,1.16,1.17,1.17,1.15,1.13,1.14,1.16,1.12,1.11,1.11,1.14,1.15,1.18,1.20,1.19,1.17


## 6. Saídas e integridade dos brutos

In [8]:
saidas = {
    "amizades_limpa.parquet": limpa,
    "rede_por_conta.parquet": rpc,
    "rede_mes.parquet": rm,
}
for nome, df in saidas.items():
    df = df.reset_index(drop=True)
    df.to_parquet(PROCESSADOS / nome, index=False)
    pd.testing.assert_frame_equal(pd.read_parquet(PROCESSADOS / nome), df)
    print(f"dados/processados/{nome}: {len(df):,} linhas x {df.shape[1]} colunas")

pd.DataFrame(registro).to_csv(SAIDAS / "limpeza_amizades_registro.csv", index=False)
display(pd.DataFrame(registro))

hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/processados/amizades_limpa.parquet: 113,207 linhas x 4 colunas


dados/processados/rede_por_conta.parquet: 64,769 linhas x 8 colunas


dados/processados/rede_mes.parquet: 327,771 linhas x 6 colunas


,tabela,passo,linhas,observacao
0,amizades,0. Arquivo bruto,113480,"0 nas checagens de completude, validade e unic..."
1,amizades,1. Até o corte,113207,273 removidas (desde > 30/09/2026)
2,amizades,2. Marca de reciprocidade,113207,"84,498 recíprocas"
3,rede_por_conta,3. Uma linha por conta,64769,"10,607 sem nenhuma relação"
4,rede_mes,4. Uma linha por linha do painel,327771,"240,568 com alguma relação"


dados/brutos/ inalterado: True
